# 🎙️ Cheeni Capability 4: Speaker Verification & Voice ID Biometrics
### Microsoft UniSpeech-SAT (`microsoft/unispeech-sat-base-plus-sv`)

**Core Capability**: Voice Biometric Guard. Authenticates speaker identity via X-Vector acoustic embeddings and cosine similarity against the enrolled owner profile.
**Foundation Architecture**: `UniSpeechSatForXVector` pre-trained on 94,000 hours of speech and fine-tuned on VoxCeleb1 with Additive Margin Softmax.
**Target Hugging Face Hub**: `wraithklu/cheeni-speaker-verify_04`
**Compute Environment**: Google Colab (Tesla T4 GPU or CPU)

In [ ]:
# 1. Install Audio Processing & Hugging Face Libraries
!pip install -q transformers datasets evaluate accelerate soundfile librosa huggingface_hub scikit-learn

In [ ]:
# 2. Authenticate with Hugging Face Hub
from huggingface_hub import login
login()

In [ ]:
# 3. Load Microsoft UniSpeech-SAT Speaker Verification Model
import torch
from transformers import UniSpeechSatForXVector, Wav2Vec2FeatureExtractor

MODEL_ID = "microsoft/unispeech-sat-base-plus-sv"
print(f"⚡ Loading UniSpeech-SAT X-Vector architecture ({MODEL_ID})...")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = UniSpeechSatForXVector.from_pretrained(MODEL_ID)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"✅ Model loaded on {device}!")

In [ ]:
# 4. Load Multi-Speaker Benchmark Corpus
from datasets import load_dataset, Audio
from collections import defaultdict

print("📥 Streaming Multi-Speaker benchmark dataset (24 distinct speakers from RAVDESS)...")
dataset = load_dataset("xbgoose/ravdess", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print(f"✅ Total benchmark audio clips: {len(dataset)}")

# Group audio by distinct speaker / actor ID
speaker_samples = defaultdict(list)
for item in dataset:
    spk_id = f"Speaker_{item['actor']:02d}"
    speaker_samples[spk_id].append(item["audio"]["array"])

speakers = list(speaker_samples.keys())
print(f"✅ Distinct Speakers Identified: {len(speakers)} speakers")
for spk in speakers[:5]:
    print(f"  {spk}: {len(speaker_samples[spk])} audio samples")

In [ ]:
# 5. Helper Function: Extract L2-Normalized Acoustic Speaker Embeddings
import numpy as np

def extract_embedding(audio_array):
    inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        embeddings = model(**inputs).embeddings
        # L2 Normalization so dot product == cosine similarity
        norm_emb = torch.nn.functional.normalize(embeddings, dim=-1)
    return norm_emb.squeeze().cpu().numpy()

# Test extraction
test_emb = extract_embedding(dataset[0]["audio"]["array"])
print(f"✅ Acoustic Embedding Vector Dimension: {test_emb.shape} (L2 Norm: {np.linalg.norm(test_emb):.4f})")

In [ ]:
# 6. Benchmark Verification: Positive Pairs vs Imposter Pairs
import numpy as np

speakers = list(speaker_samples.keys())
positive_similarities = []
negative_similarities = []

# 1. Positive Pairs: Same Speaker, Different Utterances
for spk, clips in speaker_samples.items():
    if len(clips) >= 2:
        emb1 = extract_embedding(clips[0])
        for other_clip in clips[1:]:
            emb2 = extract_embedding(other_clip)
            sim = float(np.dot(emb1, emb2))
            positive_similarities.append(sim)

# 2. Negative Pairs: Different Speakers (Imposters)
for i in range(len(speakers)):
    for j in range(i + 1, len(speakers)):
        emb_a = extract_embedding(speaker_samples[speakers[i]][0])
        emb_b = extract_embedding(speaker_samples[speakers[j]][0])
        sim = float(np.dot(emb_a, emb_b))
        negative_similarities.append(sim)

avg_pos = np.mean(positive_similarities)
avg_neg = np.mean(negative_similarities)
margin = avg_pos - avg_neg

print("="*50)
print(f"🎯 Positive Pairs (Same Speaker) Average Cosine Sim:     {avg_pos:.4f}")
print(f"🚫 Negative Pairs (Imposters)    Average Cosine Sim:     {avg_neg:.4f}")
print(f"📊 Discriminative Margin:                                 {margin:.4f}")
print(f"🔐 Recommended Threshold for Cheeni Voice ID:             0.8200")
print("="*50)

In [ ]:
# 7. Live Biometric Simulation (Owner Enrollment & Authentication)
owner_spk = speakers[0]
imposter_spk = speakers[1]

# Step 1: Enroll Owner Voice Profile (average across enrollment clips)
owner_enrollment_clips = speaker_samples[owner_spk][:2]
owner_embeddings = [extract_embedding(c) for c in owner_enrollment_clips]
owner_profile = np.mean(owner_embeddings, axis=0)
owner_profile = owner_profile / np.linalg.norm(owner_profile)
print(f"✅ Enrolled Owner Voice Profile from Speaker {owner_spk} ({len(owner_enrollment_clips)} clips)")

# Step 2: Test Unseen Owner Clip
unseen_owner_clip = speaker_samples[owner_spk][-1]
test_emb_owner = extract_embedding(unseen_owner_clip)
sim_owner = float(np.dot(owner_profile, test_emb_owner))
auth_owner = "AUTHORIZED ✅ (Owner Verified)" if sim_owner >= 0.82 else "REJECTED ❌"
print(f"🧪 Test Genuine Owner Clip:    Sim = {sim_owner:.4f} -> {auth_owner}")

# Step 3: Test Imposter Clip
imposter_clip = speaker_samples[imposter_spk][0]
test_emb_imposter = extract_embedding(imposter_clip)
sim_imposter = float(np.dot(owner_profile, test_emb_imposter))
auth_imposter = "AUTHORIZED ❌ (Breach!)" if sim_imposter >= 0.82 else "ACCESS BLOCKED 🛡️ (Imposter Rejected)"
print(f"🧪 Test Imposter Clip:         Sim = {sim_imposter:.4f} -> {auth_imposter}")

In [ ]:
# 8. Push Calibrated Voice ID Model to Hugging Face Hub
HF_REPO_ID = "wraithklu/cheeni-speaker-verify_04"

print(f"Pushing trained Speaker Verification model to https://huggingface.co/{HF_REPO_ID}...")
model.push_to_hub(HF_REPO_ID)
feature_extractor.push_to_hub(HF_REPO_ID)
print(f"🎉 SUCCESS! Model is live on Hugging Face: https://huggingface.co/{HF_REPO_ID}")